# LAB 6 — Langflow: RAG-Fusion Visual (Bônus)
## Aula 7 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Construir o pipeline RAG-Fusion visualmente no Langflow, utilizando componentes paralelos para demonstrar o fluxo multi-query de forma intuitiva. Este laboratório é classificado como **bônus** (+5 pontos).

**Tempo estimado:** 20 minutos

**O que é Langflow?** Uma plataforma visual no-code/low-code para construir pipelines LLM. Cada nó representa um componente (LLM, retriever, prompt, etc.) e as conexões representam o fluxo de dados.

---
**Checklist de entrega (bônus):**
- [ ] Langflow instalado e rodando
- [ ] Fluxo RAG-Fusion construído visualmente
- [ ] Screenshot do fluxo capturado
- [ ] Fluxo exportado como JSON

## 1. Instalação do Langflow

In [ ]:
# Instalar Langflow
!pip install -q langflow
print('✅ Langflow instalado!')

# Verificar versão
import langflow
print(f'Versão: {langflow.__version__}')

## 2. Iniciar Servidor Langflow

In [ ]:
# No Google Colab, iniciar o servidor em background
import subprocess
import threading

def run_langflow():
    subprocess.run(['python', '-m', 'langflow', 'run', '--host', '0.0.0.0', '--port', '7860'])

# Iniciar em thread separada
t = threading.Thread(target=run_langflow, daemon=True)
t.start()

import time
time.sleep(10)  # aguardar inicialização

print('✅ Langflow iniciado!')
print('Para acessar no Colab, use o link abaixo (ngrok ou localtunnel):')

In [ ]:
# Expor Langflow via ngrok (para acesso externo no Colab)
!pip install -q pyngrok
from pyngrok import ngrok

# Criar túnel
public_url = ngrok.connect(7860)
print(f'✅ Acesse o Langflow em: {public_url}')
print('\nInstruções:')
print('1. Clique no link acima para abrir o Langflow')
print('2. Crie um novo fluxo')
print('3. Siga as instruções na próxima célula')

## 3. Construindo o Fluxo RAG-Fusion no Langflow

Após abrir o Langflow no link acima, siga estas etapas para construir o pipeline RAG-Fusion visualmente:

### Arquitetura do Fluxo

```
┌─────────────────┐
│   Chat Input    │  ← Query do usuário
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  LLM Sub-query  │  ← vLLM gera 3 sub-queries
│   Generator     │
└──┬──────────┬───┘
   │          │
   ▼          ▼          ← Branches paralelos
┌──────┐  ┌──────┐  ┌──────┐
│ kNN  │  │ kNN  │  │ kNN  │  ← 3 retrievers OpenSearch
│ Q1   │  │ Q2   │  │ Q3   │
└──┬───┘  └──┬───┘  └──┬───┘
   │          │          │
   └──────────┴──────────┘
                │
                ▼
         ┌─────────┐
         │   RRF   │  ← Fusão dos rankings
         │  Merger │
         └────┬────┘
              │
              ▼
         ┌─────────┐
         │   LLM   │  ← Llama 3.1 gera resposta
         └────┬────┘
              │
              ▼
         ┌─────────┐
         │  Output │
         └─────────┘
```

### Passos no Langflow:

1. **Arrastar** componente `Chat Input` para a tela
2. **Adicionar** componente `OpenAI` → configurar com URL do vLLM
3. **Criar** prompt template para geração de sub-queries
4. **Duplicar** o nó de retrieval OpenSearch 3 vezes (um por sub-query)
5. **Conectar** as saídas dos 3 retrievers ao componente `Combine Text`
6. **Adicionar** nó LLM final para geração da resposta
7. **Conectar** ao `Chat Output`

## 4. Exportar e Importar Fluxo via API

In [ ]:
# Template de fluxo RAG-Fusion para importar no Langflow
# Copie este JSON e importe em Langflow > Import > Paste JSON

rag_fusion_flow = {
  "name": "RAG-Fusion Jurídico",
  "description": "Pipeline RAG-Fusion com 3 sub-queries paralelas para corpus jurídico",
  "nodes": [
    {
      "id": "chat_input",
      "type": "ChatInput",
      "position": {"x": 100, "y": 300},
      "data": {"input_value": ""}
    },
    {
      "id": "subquery_generator",
      "type": "OpenAIModel",
      "position": {"x": 350, "y": 300},
      "data": {
        "base_url": "http://localhost:8000/v1",
        "model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "temperature": 0.5,
        "system_message": "Gere 3 sub-queries para o corpus jurídico. Retorne uma por linha."
      }
    },
    {
      "id": "retriever_1",
      "type": "OpenSearchRetriever",
      "position": {"x": 650, "y": 100},
      "data": {"index": "corpus_juridico_aula7", "k": 5}
    },
    {
      "id": "retriever_2",
      "type": "OpenSearchRetriever",
      "position": {"x": 650, "y": 300},
      "data": {"index": "corpus_juridico_aula7", "k": 5}
    },
    {
      "id": "retriever_3",
      "type": "OpenSearchRetriever",
      "position": {"x": 650, "y": 500},
      "data": {"index": "corpus_juridico_aula7", "k": 5}
    },
    {
      "id": "rrf_merger",
      "type": "CombineText",
      "position": {"x": 950, "y": 300},
      "data": {"separator": "\n\n---\n\n"}
    },
    {
      "id": "final_llm",
      "type": "OpenAIModel",
      "position": {"x": 1200, "y": 300},
      "data": {
        "base_url": "http://localhost:8000/v1",
        "model_name": "meta-llama/Llama-3.1-8B-Instruct",
        "temperature": 0.1,
        "system_message": "Responda baseado nos documentos recuperados. Seja preciso e cite as fontes."
      }
    },
    {
      "id": "chat_output",
      "type": "ChatOutput",
      "position": {"x": 1450, "y": 300}
    }
  ]
}

import json
print('=== TEMPLATE DO FLUXO RAG-FUSION (Langflow JSON) ===')
print('Copie o JSON abaixo e importe no Langflow:')
print()
print(json.dumps(rag_fusion_flow, indent=2, ensure_ascii=False)[:500] + '...')
print()

# Salvar template
with open('rag_fusion_langflow_template.json', 'w', encoding='utf-8') as f:
    json.dump(rag_fusion_flow, f, indent=2, ensure_ascii=False)

print('✅ Template salvo em: rag_fusion_langflow_template.json')

## 5. Capturar Screenshot do Fluxo

Após construir o fluxo no Langflow:

1. **Capturar screenshot** da tela do Langflow mostrando o fluxo completo
2. **Fazer upload** da imagem nesta célula (arraste ou use o ícone de imagem)
3. **Adicionar** um comentário descrevendo o que cada nó faz

**Cole ou importe o screenshot aqui:**

### Descrição do Fluxo Construído

Descreva abaixo:
- Quais componentes você usou?
- Como conectou os nós de retrieval em paralelo?
- Qual foi a maior dificuldade?
- O Langflow facilitou a compreensão visual do pipeline em comparação com o código Python?

> **Sua descrição aqui:**

In [ ]:
print('=== CHECKLIST DE ENTREGA — LAB 6 (BÔNUS) ===')
checklist = [
    'Langflow instalado e servidor iniciado',
    'Fluxo RAG-Fusion construído (mínimo: Input → LLM → 2 Retrievers → Merge → LLM → Output)',
    'Screenshot do fluxo capturado e inserido no notebook',
    'Template JSON exportado (rag_fusion_langflow_template.json)',
    'Descrição do fluxo e reflexão escritas'
]
for item in checklist:
    print(f'  [ ] {item}')
print('\n🎉 Parabéns! Aula 7 concluída!')
print('Próxima aula: Aula 8 — Self-RAG, CRAG e LangGraph')